Cell 1: Importeer Benodigde Libraries


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf

Cell 2: Declareer Constanten

In [ ]:
DATA_DIR = "data" 

IMAGE_SIZE = 100

Cell 3: Laad and Verwerk Foto's

In [ ]:
images = []
genders = []

for filename in os.listdir(DATA_DIR):
    if filename.endswith(".jpg"):
        try:
            gender = int(filename.split("_")[1])
            img_path = os.path.join(DATA_DIR, filename)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            img = img / 255.0
            images.append(img)
            genders.append(gender)
        except Exception as e:
            print(f"Fout met laden van {filename}: {e}")


Error loading 53__0_20170116184028385.jpg: invalid literal for int() with base 10: ''


Cell 4: Zet over naar NumPy Arrays

In [ ]:
images = np.array(images, dtype=np.float32)
genders = np.array(genders, dtype=np.float32)

print(f"Foto shape: {images.shape}")
print(f"Geslachten shape: {genders.shape}")

Images shape: (24105, 100, 100, 3)
Ages shape: (24105,)


Cell 5: Split in Training en Test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(images, genders, test_size=0.2, random_state=42)

Cell 6: Bouw het CNN Model

In [ ]:
model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
model.add(tf.keras.layers.MaxPooling2D((2, 2)))

model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D((2, 2)))

model.add(tf.keras.layers.Conv2D(128, (3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D((2, 2)))

model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


model.summary()

c:\Users\deanj\Documents\VSCODE\Neural\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       819,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 912,577 (3.48 MB)

 Trainable params: 912,577 (3.48 MB)

 Non-trainable params: 0 (0.00 B)

Cell 7: Train het Model 

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    callbacks=[early_stopping]
)

model.save('gender_model.keras')

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - accuracy: 0.6065 - loss: 0.6493 - val_accuracy: 0.6604 - val_loss: 0.5948
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 50s 92ms/step - accuracy: 0.7334 - loss: 0.5261 - val_accuracy: 0.7413 - val_loss: 0.5181
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 49s 91ms/step - accuracy: 0.7744 - loss: 0.4668 - val_accuracy: 0.7439 - val_loss: 0.4972
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 49s 91ms/step - accuracy: 0.7895 - loss: 0.4330 - val_accuracy: 0.7724 - val_loss: 0.4735
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 50s 91ms/step - accuracy: 0.8186 - loss: 0.3946 - val_accuracy: 0.7709 - val_loss: 0.4760
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 56s 104ms/step - accuracy: 0.8427 - loss: 0.3467 - val_accuracy: 0.7864 - val_loss: 0.4875
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 52s 95ms/step - accuracy: 0.8682 - loss: 0.2931 - val_accuracy: 0.7999 - val_loss: 0.4700
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 51s 95ms/step - accuracy: 0.8932 - loss: 0.2474 -

Cell 8: Plot Train Progressie


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"\nTest Nauwkeurigheid: {accuracy * 100:.2f}%")

151/151 - 4s - 27ms/step - accuracy: 0.7903 - loss: 0.4453

Test Accuracy: 79.03%


Cell 9: Test op een aantal foto's


In [ ]:
predictions = model.predict(X_test[:10])
for i in range(10):
    actual = int(y_test[i])
    predicted = int(predictions[i][0] > 0.5)
    print(f"Actueel: {'Man' if actual == 0 else 'Vrouw'} | Voorspelling: {'Man' if predicted == 0 else 'Vrouw'}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Actual: Male | Predicted: Male
Actual: Male | Predicted: Male
Actual: Female | Predicted: Male
Actual: Female | Predicted: Female
Actual: Male | Predicted: Male
Actual: Male | Predicted: Female
Actual: Male | Predicted: Male
Actual: Female | Predicted: Female
Actual: Male | Predicted: Male
Actual: Male | Predicted: Male


Cell 10: Test op je eigen gezicht

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import os

model = tf.keras.models.load_model('gender_model.keras')

IMAGE_SIZE = 100

def predict_gender_from_file(img_path):
    img = cv2.imread(img_path)
    
    if img is None:
        raise FileNotFoundError(f"Kan foto niet laden van pad: {img_path}")
    
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    
    img = img / 255.0
    
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)
    
    gender = "Vrouw" if prediction[0][0] > 0.5 else "Man"
    return gender

folder_path = 'custom_test'

for filename in os.listdir(folder_path):
    img_path = os.path.join(folder_path, filename)

    if img_path.endswith(('.jpg', '.jpeg', '.png')):
        try:
            predicted_gender = predict_gender_from_file(img_path)
            print(f"Foto: {filename} -> Gerande Geslacht: {predicted_gender}")
        except FileNotFoundError as e:
            print(e)
        except Exception as e:
            print(f"Fout met verwerken van foto {filename}: {e}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
Image: WhatsApp Image 2025-05-03 at 00.15.43_3304c7bb.jpg -> Predicted Gender: Male
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Image: WhatsApp Image 2025-05-03 at 00.15.43_db71a8d3.jpg -> Predicted Gender: Male
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_08f4e177.jpg -> Predicted Gender: Female
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_0acc298b.jpg -> Predicted Gender: Female
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_5799d19d.jpg -> Predicted Gender: Female
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_cc367bd0.jpg -> Predicted Gender: Female
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_dfc55006.jpg -> Predicted Gender: Female
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Image: WhatsApp Image 2025-05-03 at 01.02.54_e2cbdc07.jpg -> Predicted Gender: Male
1/1 ━━━━━━━━━